# 🚀 SPEC-03: Aspect-Based Sentiment Analysis (ABSA) — Model Training & Evaluation
## 📌 IndoRoBERTa Classifier (110M) vs. SahabatAI Classifier (8B QLoRA)

- **Target Models:**
  1. **IndoRoBERTa-base** (`indolem/indobert-base-uncased`, 110M) — Multi-Head Classifier (Full Fine-Tuning)
  2. **SahabatAI-Instruct-8B** (`SahabatAI/SahabatAI-Instruct-8B`, 8B) — Multi-Head Classifier (4-bit NF4 QLoRA Partial Fine-Tuning)
- **Dataset:** 100% Pure Human Gold Standard (667 Train / 287 Validation, Stratified 70/30)
- **Environment:** Kaggle Notebook Free Tier (NVIDIA T4 GPU 16GB VRAM)
- **Training Epochs:** 5 Epochs per model
- **OOM Safeguards:** 4-bit NF4 Quantization, Gradient Checkpointing, Micro-Batching + Gradient Accumulation, FP16 Mixed Precision, Explicit CUDA Memory Cleaning, Max Sequence Length 128 Tokens.


## 1. Setup Dependencies & Environment Imports

In [ ]:
# Install required packages for Kaggle T4 GPU
!pip install -q -U transformers peft bitsandbytes accelerate scikit-learn seaborn matplotlib tqdm

import os
import gc
import json
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Check CUDA availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device Utama: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"💾 Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set Random Seed for Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

LABEL2ID = {'None': 0, 'none': 0, 'positif': 1, 'netral': 2, 'negatif': 3}
ID2LABEL = {0: 'None', 1: 'positif', 2: 'netral', 3: 'negatif'}
ASPECTS = ['infra', 'ekonomi', 'kualitas', 'purnajual']
CLASS_NAMES = ['None', 'Positif', 'Netral', 'Negatif']


## 2. Load Dataset (Pure Human Gold Standard 70/30 Split)

In [ ]:
# Detect dataset directory in Kaggle environment or local fallback
data_dir = None
possible_paths = [
    Path("/kaggle/input/indonesian-ev-absa-dataset"),
    Path("/kaggle/input"),
    Path("data/processed"),
    Path("../data/processed")
]

for p in possible_paths:
    if p.exists():
        if (p / "train.csv").exists():
            data_dir = p
            break
        # Search subdirectories inside /kaggle/input
        matches = list(p.glob("**/train.csv"))
        if matches:
            data_dir = matches[0].parent
            break

if data_dir is None:
    raise FileNotFoundError("❌ File train.csv tidak ditemukan! Harap pastikan dataset sudah di-upload ke Kaggle.")

print(f"📂 Mengakses Dataset dari: {data_dir.resolve()}")

train_df = pd.read_csv(data_dir / "train.csv", encoding="utf-8-sig")
val_df = pd.read_csv(data_dir / "val.csv", encoding="utf-8-sig")

print(f"✅ Train Set : {len(train_df)} baris")
print(f"✅ Val Set   : {len(val_df)} baris")
display(train_df.head(3))


## 3. Class Weights Calculation & PyTorch Dataset Definition

In [ ]:
# 1. Compute Loss Class Weights to address imbalance
def compute_aspect_class_weights(df, aspects=ASPECTS):
    weights_dict = {}
    classes = np.array([0, 1, 2, 3])
    for aspect in aspects:
        col = f"{aspect}_sentiment"
        raw_vals = df[col].fillna("none").astype(str).str.lower().map(LABEL2ID).fillna(0).astype(int).values
        weights = compute_class_weight(class_weight="balanced", classes=classes, y=raw_vals)
        weights_dict[aspect] = torch.tensor(weights, dtype=torch.float32).to(device)
        print(f"  • Weight {aspect:10s}: {weights.round(3)}")
    return weights_dict

print("⚖️ Loss Class Weights per Aspek:")
class_weights = compute_aspect_class_weights(train_df)

# 2. PyTorch ABSADataset Wrapper
class ABSADataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128, aspects=ASPECTS):
        # Auto-detect column containing text (text_cleaned, text_original, or comment_text)
        if "text_cleaned" in df.columns:
            text_col = "text_cleaned"
        elif "text_original" in df.columns:
            text_col = "text_original"
        elif "comment_text" in df.columns:
            text_col = "comment_text"
        else:
            raise KeyError(f"❌ Tidak dapat menemukan kolom teks di DataFrame. Kolom tersedia: {list(df.columns)}")

        self.texts = df[text_col].fillna("").astype(str).tolist()
        self.labels = {}
        for aspect in aspects:
            col = f"{aspect}_sentiment"
            raw = df[col].fillna("none").astype(str).str.lower().map(LABEL2ID).fillna(0).astype(int).values
            self.labels[aspect] = torch.tensor(raw, dtype=torch.long)
        
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.aspects = aspects

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": {aspect: self.labels[aspect][idx] for aspect in self.aspects}
        }
        return item


## 4. Metrics Evaluator & Plotting Functions

In [ ]:
def evaluate_predictions(y_true_dict, y_pred_dict, aspects=ASPECTS):
    aspect_metrics = {}
    cm_dict = {}
    macro_f1s, weighted_f1s, accuracies = [], [], []

    for aspect in aspects:
        y_true = np.array(y_true_dict[aspect])
        y_pred = np.array(y_pred_dict[aspect])

        acc = accuracy_score(y_true, y_pred)
        prec_m = precision_score(y_true, y_pred, average="macro", zero_division=0)
        rec_m = recall_score(y_true, y_pred, average="macro", zero_division=0)
        f1_m = f1_score(y_true, y_pred, average="macro", zero_division=0)

        prec_w = precision_score(y_true, y_pred, average="weighted", zero_division=0)
        rec_w = recall_score(y_true, y_pred, average="weighted", zero_division=0)
        f1_w = f1_score(y_true, y_pred, average="weighted", zero_division=0)

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])

        aspect_metrics[aspect] = {
            "accuracy": acc,
            "precision_macro": prec_m,
            "recall_macro": rec_m,
            "f1_macro": f1_m,
            "precision_weighted": prec_w,
            "recall_weighted": rec_w,
            "f1_weighted": f1_w,
        }
        cm_dict[aspect] = cm
        accuracies.append(acc)
        macro_f1s.append(f1_m)
        weighted_f1s.append(f1_w)

    n_samples = len(y_true_dict[aspects[0]])
    exact_matches = sum(
        all(y_true_dict[asp][i] == y_pred_dict[asp][i] for asp in aspects)
        for i in range(n_samples)
    )
    exact_match_ratio = exact_matches / n_samples if n_samples > 0 else 0.0

    return {
        "per_aspect": aspect_metrics,
        "overall": {
            "mean_accuracy": np.mean(accuracies),
            "mean_macro_f1": np.mean(macro_f1s),
            "mean_weighted_f1": np.mean(weighted_f1s),
            "exact_match_ratio": exact_match_ratio,
        },
        "confusion_matrices": cm_dict,
    }

def plot_confusion_matrices(cm_dict, model_name, save_path, aspects=ASPECTS):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f"Confusion Matrices 4x4 — {model_name}", fontsize=16, fontweight="bold", y=0.98)

    for idx, aspect in enumerate(aspects):
        ax = axes[idx // 2, idx % 2]
        cm = cm_dict[aspect]
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        ax.set_title(f"Aspek: {aspect.capitalize()}", fontsize=14, fontweight="semibold")
        ax.set_xlabel("Predicted Label", fontsize=11)
        ax.set_ylabel("True Label", fontsize=11)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    return save_path


## 5. Model 1: IndoRoBERTa Multi-Head Classifier Training (5 Epochs)

In [ ]:
from transformers import AutoModel, AutoTokenizer

class IndoRoBERTaMultiHeadClassifier(nn.Module):
    def __init__(self, model_name="indolem/indobert-base-uncased", num_classes=4, dropout_prob=0.2, aspects=ASPECTS):
        super().__init__()
        self.aspects = aspects
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.heads = nn.ModuleDict({
            aspect: nn.Sequential(
                nn.Dropout(dropout_prob),
                nn.Linear(hidden_size, num_classes)
            ) for aspect in aspects
        })

    def forward(self, input_ids, attention_mask, labels=None, class_weights=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.pooler_output if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None else outputs.last_hidden_state[:, 0, :]

        logits = {}
        total_loss = 0.0

        for aspect in self.aspects:
            aspect_logits = self.heads[aspect](cls_output)
            logits[aspect] = aspect_logits

            if labels is not None:
                target = labels[aspect]
                weight = class_weights[aspect] if class_weights and aspect in class_weights else None
                criterion = nn.CrossEntropyLoss(weight=weight)
                total_loss += criterion(aspect_logits, target)

        output_dict = {"logits": logits}
        if labels is not None:
            output_dict["loss"] = total_loss
        return output_dict

# Tokenizer & Dataloaders
indobert_model_name = "indolem/indobert-base-uncased"
indobert_tokenizer = AutoTokenizer.from_pretrained(indobert_model_name)

train_dataset_roberta = ABSADataset(train_df, indobert_tokenizer, max_len=128)
val_dataset_roberta = ABSADataset(val_df, indobert_tokenizer, max_len=128)

train_loader_roberta = DataLoader(train_dataset_roberta, batch_size=16, shuffle=True)
val_loader_roberta = DataLoader(val_dataset_roberta, batch_size=16, shuffle=False)

# Model Initialization
indobert_model = IndoRoBERTaMultiHeadClassifier(indobert_model_name).to(device)
optimizer_roberta = torch.optim.AdamW(indobert_model.parameters(), lr=2e-5, weight_decay=0.01)
epochs = 5

print("🏋️ Starting Training: IndoRoBERTa Classifier (5 Epochs)...")

for epoch in range(1, epochs + 1):
    indobert_model.train()
    total_train_loss = 0.0
    for batch in tqdm(train_loader_roberta, desc=f"IndoRoBERTa Epoch {epoch}/{epochs}"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = {k: v.to(device) for k, v in batch["labels"].items()}

        optimizer_roberta.zero_grad()
        output = indobert_model(input_ids, attention_mask, labels=labels, class_weights=class_weights)
        loss = output["loss"]
        loss.backward()
        optimizer_roberta.step()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader_roberta)
    print(f"  Epoch {epoch} | Loss Latih: {avg_train_loss:.4f}")

# Evaluation on Validation Set
indobert_model.eval()
y_true_roberta = {asp: [] for asp in ASPECTS}
y_pred_roberta = {asp: [] for asp in ASPECTS}

with torch.no_grad():
    for batch in val_loader_roberta:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        output = indobert_model(input_ids, attention_mask)
        logits = output["logits"]

        for asp in ASPECTS:
            preds = torch.argmax(logits[asp], dim=1).cpu().numpy()
            trues = labels[asp].numpy()
            y_pred_roberta[asp].extend(preds)
            y_true_roberta[asp].extend(trues)

roberta_results = evaluate_predictions(y_true_roberta, y_pred_roberta)
print("
📊 HASIL EVALUASI INDOROBERTA CLASSIFIER:")
print(f"  • Mean Macro F1-Score: {roberta_results['overall']['mean_macro_f1']:.4f}")
print(f"  • Exact Match Ratio  : {roberta_results['overall']['exact_match_ratio']:.4f}")
print(f"  • Mean Accuracy     : {roberta_results['overall']['mean_accuracy']:.4f}")

# Save Checkpoint & Confusion Matrix
os.makedirs("models/indoroberta_absa", exist_ok=True)
torch.save(indobert_model.state_dict(), "models/indoroberta_absa/pytorch_model.bin")
indobert_tokenizer.save_pretrained("models/indoroberta_absa")
plot_confusion_matrices(roberta_results["confusion_matrices"], "IndoRoBERTa Classifier", "confusion_matrices_indoroberta.png")


## 6. VRAM Protection & Memory Garbage Collection

In [ ]:
# Explicitly release VRAM before loading SahabatAI-8B
print("🧹 Membersihkan memori VRAM sebelum memuat SahabatAI-8B...")
del indobert_model
del optimizer_roberta
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"💾 VRAM Tersisa: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB (Allocated) / {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB (Total)")


## 7. Model 2: SahabatAI-8B Multi-Head Classifier Training (5 Epochs QLoRA)

In [ ]:
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

class SahabatAIMultiHeadClassifier(nn.Module):
    def __init__(self, model_name="SahabatAI/SahabatAI-Instruct-8B", num_classes=4, dropout_prob=0.1, aspects=ASPECTS):
        super().__init__()
        self.aspects = aspects

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        self.backbone = AutoModel.from_pretrained(
            model_name,
            quantization_config=bnb_config,
            device_map="auto",
            trust_remote_code=True,
            torch_dtype=torch.float16,
        )

        hidden_size = self.backbone.config.hidden_size
        self.backbone = prepare_model_for_kbit_training(self.backbone)

        peft_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
            lora_dropout=0.05,
            bias="none",
            task_type="FEATURE_EXTRACTION",
        )
        self.backbone = get_peft_model(self.backbone, peft_config)

        self.heads = nn.ModuleDict({
            aspect: nn.Sequential(
                nn.Dropout(dropout_prob),
                nn.Linear(hidden_size, num_classes)
            ) for aspect in aspects
        })

    def forward(self, input_ids, attention_mask, labels=None, class_weights=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state

        batch_size = input_ids.shape[0]
        sequence_lengths = attention_mask.sum(dim=1) - 1
        last_token_hidden = last_hidden_state[torch.arange(batch_size), sequence_lengths]

        logits = {}
        total_loss = 0.0

        for aspect in self.aspects:
            head = self.heads[aspect].to(last_token_hidden.device)
            aspect_logits = head(last_token_hidden)
            logits[aspect] = aspect_logits

            if labels is not None:
                target = labels[aspect].to(aspect_logits.device)
                weight = class_weights[aspect].to(aspect_logits.device) if class_weights and aspect in class_weights else None
                criterion = nn.CrossEntropyLoss(weight=weight)
                total_loss += criterion(aspect_logits, target)

        output_dict = {"logits": logits}
        if labels is not None:
            output_dict["loss"] = total_loss
        return output_dict

# Load SahabatAI Tokenizer
sahabatai_model_name = "SahabatAI/SahabatAI-Instruct-8B"
sahabatai_tokenizer = AutoTokenizer.from_pretrained(sahabatai_model_name, trust_remote_code=True)
if sahabatai_tokenizer.pad_token is None:
    sahabatai_tokenizer.pad_token = sahabatai_tokenizer.eos_token
sahabatai_tokenizer.padding_side = "left"

train_dataset_sahabat = ABSADataset(train_df, sahabatai_tokenizer, max_len=128)
val_dataset_sahabat = ABSADataset(val_df, sahabatai_tokenizer, max_len=128)

# Micro batch 2 + Grad Accum 8 = Effective Batch Size 16 (OOM Safeguard)
train_loader_sahabat = DataLoader(train_dataset_sahabat, batch_size=2, shuffle=True)
val_loader_sahabat = DataLoader(val_dataset_sahabat, batch_size=2, shuffle=False)

# Initialize Model
sahabatai_model = SahabatAIMultiHeadClassifier(sahabatai_model_name)
optimizer_sahabat = torch.optim.AdamW(sahabatai_model.parameters(), lr=2e-4, weight_decay=0.01)
grad_accum_steps = 8
epochs = 5

print("🏋️ Starting Training: SahabatAI-8B QLoRA Classifier (5 Epochs)...")

for epoch in range(1, epochs + 1):
    sahabatai_model.train()
    total_train_loss = 0.0
    optimizer_sahabat.zero_grad()

    for step, batch in enumerate(tqdm(train_loader_sahabat, desc=f"SahabatAI Epoch {epoch}/{epochs}")):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        output = sahabatai_model(input_ids, attention_mask, labels=labels, class_weights=class_weights)
        loss = output["loss"] / grad_accum_steps
        loss.backward()

        if (step + 1) % grad_accum_steps == 0 or (step + 1) == len(train_loader_sahabat):
            optimizer_sahabat.step()
            optimizer_sahabat.zero_grad()

        total_train_loss += loss.item() * grad_accum_steps

    avg_train_loss = total_train_loss / len(train_loader_sahabat)
    print(f"  Epoch {epoch} | Loss Latih: {avg_train_loss:.4f}")

# Evaluation on Validation Set
sahabatai_model.eval()
y_true_sahabat = {asp: [] for asp in ASPECTS}
y_pred_sahabat = {asp: [] for asp in ASPECTS}

with torch.no_grad():
    for batch in val_loader_sahabat:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        output = sahabatai_model(input_ids, attention_mask)
        logits = output["logits"]

        for asp in ASPECTS:
            preds = torch.argmax(logits[asp], dim=1).cpu().numpy()
            trues = labels[asp].numpy()
            y_pred_sahabat[asp].extend(preds)
            y_true_sahabat[asp].extend(trues)

sahabat_results = evaluate_predictions(y_true_sahabat, y_pred_sahabat)
print("
📊 HASIL EVALUASI SAHABATAI-8B QLORA CLASSIFIER:")
print(f"  • Mean Macro F1-Score: {sahabat_results['overall']['mean_macro_f1']:.4f}")
print(f"  • Exact Match Ratio  : {sahabat_results['overall']['exact_match_ratio']:.4f}")
print(f"  • Mean Accuracy     : {sahabat_results['overall']['mean_accuracy']:.4f}")

# Save Checkpoint & Confusion Matrix
os.makedirs("models/sahabatai_absa_lora", exist_ok=True)
sahabatai_model.backbone.save_pretrained("models/sahabatai_absa_lora")
sahabatai_tokenizer.save_pretrained("models/sahabatai_absa_lora")
plot_confusion_matrices(sahabat_results["confusion_matrices"], "SahabatAI-8B Classifier", "confusion_matrices_sahabatai.png")


## 8. Comparative Analysis & Summary Metrics

In [ ]:
comparison_data = []

for asp in ASPECTS:
    comparison_data.append({
        "Aspek": asp.capitalize(),
        "IndoRoBERTa F1-Macro": roberta_results["per_aspect"][asp]["f1_macro"],
        "SahabatAI-8B F1-Macro": sahabat_results["per_aspect"][asp]["f1_macro"],
        "IndoRoBERTa Accuracy": roberta_results["per_aspect"][asp]["accuracy"],
        "SahabatAI-8B Accuracy": sahabat_results["per_aspect"][asp]["accuracy"],
    })

comparison_df = pd.DataFrame(comparison_data)

print("🏆 PERBANDINGAN PER ASPEK (INDOBERTA VS SAHABATAI-8B):")
display(comparison_df)

overall_summary = pd.DataFrame([
    {
        "Model": "IndoRoBERTa Classifier (110M)",
        "Mean Macro F1": roberta_results["overall"]["mean_macro_f1"],
        "Exact Match Ratio": roberta_results["overall"]["exact_match_ratio"],
        "Mean Accuracy": roberta_results["overall"]["mean_accuracy"],
    },
    {
        "Model": "SahabatAI-8B QLoRA Classifier (8B)",
        "Mean Macro F1": sahabat_results["overall"]["mean_macro_f1"],
        "Exact Match Ratio": sahabat_results["overall"]["exact_match_ratio"],
        "Mean Accuracy": sahabat_results["overall"]["mean_accuracy"],
    }
])

print("
🌟 RINGKASAN PERBANDINGAN GLOBAL:")
display(overall_summary)

# Save evaluation CSV & JSON
comparison_df.to_csv("model_comparison_per_aspect.csv", index=False)
overall_summary.to_csv("model_comparison_metrics.csv", index=False)

summary_json = {
    "indoroberta": roberta_results,
    "sahabatai": sahabat_results
}
with open("model_comparison_metrics.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2, default=str)

print("✅ Metrik perbandingan berhasil disimpan ke file CSV & JSON.")


## 9. Zip & Package Deployment Artifacts for Download

In [ ]:
def zip_dir(source_dir, zip_filename):
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as ziph:
        for root, dirs, files in os.walk(source_dir):
            for file in files:
                filepath = os.path.join(root, file)
                arcname = os.path.relpath(filepath, source_dir)
                ziph.write(filepath, arcname)
    print(f"📦 Zipped '{source_dir}' -> '{zip_filename}' ({os.path.getsize(zip_filename) / 1e6:.2f} MB)")

zip_dir("models/indoroberta_absa", "indoroberta_absa_model.zip")
zip_dir("models/sahabatai_absa_lora", "sahabatai_absa_lora.zip")

print("
🎉 SELESAI! Berkas berikut siap diunduh dari Kaggle Output:")
print("  1. indoroberta_absa_model.zip (~440 MB)")
print("  2. sahabatai_absa_lora.zip (~40 MB)")
print("  3. model_comparison_metrics.csv & JSON")
print("  4. confusion_matrices_indoroberta.png & confusion_matrices_sahabatai.png")
